# Notebook 6b: What makes a resume "Good Fit"? (label-artifact analysis)

**Runtime:** CPU is enough (no GPU needed). Run time about 5 minutes.

Notebook 3b found that a **resume-prior** baseline, which never reads the job, separates *Good Fit* from *Potential Fit* (AUC 0.727), while every content method sits at chance (about 0.5). This notebook asks **why**, with four tests:

1. **Variance decomposition.** How much of the label is explained by *which resume* it is vs *which job* it is?
2. **Label purity.** Do resumes keep the same fit level across jobs more often than chance? The null model shuffles labels *within each job*, so every job keeps its own label mix.
3. **Resume text alone.** Can a classifier that sees only the resume (never the job) predict Good vs Potential for **resumes it has never seen**? Its top-weighted words show what the labels reward.
4. **Surface features.** Length, stated years of experience, degree level: do simple resume properties track the label?

If tests 1–3 all point to the resume, the fine-grained label measures *resume quality*, not *resume–job fit*. That is the paper's RQ3 finding.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import roc_auc_score, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
SEED = 42

all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet').assign(split=s) for s in ('train', 'val', 'test')],
                   ignore_index=True)
fit = all_df[all_df.relevance > 0].copy()
fit['good'] = (fit.relevance == 2).astype(int)
print(f'{len(all_df)} pairs, {all_df.job_id.nunique()} jobs, {all_df.resume_id.nunique()} resumes')
print('label counts:', all_df.relevance.value_counts().sort_index().to_dict(), '(0 = No, 1 = Potential, 2 = Good)')
print(f'fit pairs (Potential or Good): {len(fit)}, Good share {fit.good.mean():.3f}')
print('pairs per resume: median', all_df.groupby('resume_id').size().median(),
      '| pairs per job: median', all_df.groupby('job_id').size().median())

## 1. Variance decomposition: resume vs job

We predict the label from one-hot **resume identity**, **job identity**, or both, with a nearly unregularised linear model.
- **In-sample R²** is the share of variance each factor *can* explain.
- **Cross-validated R²** (random 5-fold over pairs) is what it explains for *new pairs* of already-seen resumes and jobs. This is exactly the situation in the standard split, where 430 of 434 test resumes also appear in training.

Done twice: for the full 3-level label, and for **Good vs Potential among fit pairs** (the part content methods can't do).

In [ ]:
def decompose(df, y_col):
    y = df[y_col].to_numpy(dtype=float)
    enc = {k: OneHotEncoder(handle_unknown='ignore').fit(df[[k]]) for k in ('resume_id', 'job_id')}
    X = {'resume only': enc['resume_id'].transform(df[['resume_id']]),
         'job only': enc['job_id'].transform(df[['job_id']])}
    X['resume + job'] = sparse.hstack([X['resume only'], X['job only']]).tocsr()
    rows = []
    for name, Xm in X.items():
        m = Ridge(alpha=1e-3).fit(Xm, y)
        ins = r2_score(y, m.predict(Xm))
        pred = np.zeros_like(y)
        for tr, te in KFold(5, shuffle=True, random_state=SEED).split(y):
            enc_cv = {k: OneHotEncoder(handle_unknown='ignore').fit(df.iloc[tr][[k]]) for k in ('resume_id', 'job_id')}
            parts = [enc_cv[k].transform(df[[k]]) for k in ('resume_id', 'job_id')
                     if (k == 'resume_id' and name != 'job only') or (k == 'job_id' and name != 'resume only')]
            Xc = sparse.hstack(parts).tocsr()
            # alpha=1 here: unseen levels fall back to the intercept, and seen levels shrink a little
            pred[te] = Ridge(alpha=1.0).fit(Xc[tr], y[tr]).predict(Xc[te])
        rows.append({'target': y_col, 'factors': name, 'levels': Xm.shape[1],
                     'in-sample R2': round(ins, 3), 'CV R2 (new pairs)': round(r2_score(y, pred), 3)})
    return rows

# Within-job version: keep jobs that have both Good and Potential, and subtract each job's Good rate,
# so only the ranking *inside* a job is left (what NDCG and per-job AUC measure).
mixed = fit.groupby('job_id').good.transform(lambda g: 0 < g.mean() < 1)
within = fit[mixed].copy()
within['good (within job)'] = within.good - within.groupby('job_id').good.transform('mean')
print(f"{fit.groupby('job_id').good.mean().between(0, 1, inclusive='neither').sum()} of {fit.job_id.nunique()} jobs "
      f"have both Good and Potential; {len(within)} fit pairs in them")
job_rate = fit.groupby('job_id').good.mean()
print(f"jobs whose fit pairs are ALL Good: {(job_rate == 1).sum()}, ALL Potential: {(job_rate == 0).sum()}")

decomp = pd.DataFrame(decompose(all_df, 'relevance') + decompose(fit, 'good')
                      + [r for r in decompose(within, 'good (within job)') if r['factors'] == 'resume only'])

decomp

**How to read it:** the `good` rows mix two things. **Between jobs**: some jobs hand out mostly Good, others mostly Potential, so *job only* explains a lot. That is a job-level artifact of its own, but it cannot affect ranking inside a job. **Within a job**: the last row removes each job's Good rate and asks how much of what is left the resume identity explains. That is the part that decides NDCG and per-job AUC.

## 2. Label purity: do resumes keep their level across jobs?

For every resume with at least 2 fit pairs, **purity** is the share of its fit pairs that carry its majority level (1.0 = always Good, or always Potential).

**Null model:** shuffle the Good/Potential labels **within each job** 1,000 times. This keeps every job's own Good rate, but breaks any link to the resume. If observed purity is far above the null, the labels follow the resume.

In [ ]:
def mean_purity(df):
    g = df.groupby('resume_id').good.agg(['mean', 'size'])
    g = g[g['size'] >= 2]
    return float(np.maximum(g['mean'], 1 - g['mean']).mean()), len(g)

obs, n_res = mean_purity(fit)
rng = np.random.default_rng(SEED)
null = []
shuf = fit[['resume_id', 'job_id', 'good']].copy()
job_idx = list(shuf.groupby('job_id').indices.values())
good = shuf.good.to_numpy()
for _ in range(1000):
    g2 = good.copy()
    for idx in job_idx:
        g2[idx] = rng.permutation(g2[idx])
    shuf['good'] = g2
    null.append(mean_purity(shuf)[0])
null = np.array(null)
p_purity = (np.sum(null >= obs) + 1) / (len(null) + 1)
print(f'{n_res} resumes with >= 2 fit pairs')
print(f'observed mean purity {obs:.3f} | null (shuffled within job) {null.mean():.3f} '
      f'[{np.percentile(null, 2.5):.3f}, {np.percentile(null, 97.5):.3f}] | p = {p_purity:.4f}')

per_res = fit.groupby('resume_id').good.agg(['mean', 'size'])
per_res = per_res[per_res['size'] >= 2]
share_pure = float(((per_res['mean'] == 0) | (per_res['mean'] == 1)).mean())
print(f'resumes whose fit pairs are ALL Good or ALL Potential: {share_pure:.1%}')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(per_res['mean'], bins=20, color='#4C72B0')
ax[0].set_xlabel("Resume's share of Good among its fit pairs"); ax[0].set_ylabel('resumes')
ax[0].set_title('Most resumes are all-Good or all-Potential')
ax[1].hist(null, bins=30, color='#999999', label='null (shuffled within job)')
ax[1].axvline(obs, color='#C44E52', lw=2, label=f'observed {obs:.3f}')
ax[1].set_xlabel('mean purity'); ax[1].legend(); ax[1].set_title('Purity vs null')
plt.tight_layout(); plt.savefig(f'{RESULTS}/06b_purity.png', dpi=200); plt.show()

## 3. Resume text alone predicts Good vs Potential?

A logistic regression on the **resume's TF-IDF only**: no job text, no resume identity. It is evaluated with **GroupKFold by resume**, so every prediction is for a resume the model never saw in training. Per-job AUC is then computed exactly as in Notebook 3b, so it compares directly with the resume-prior (0.727) and the content methods (about 0.5).

For contrast, the same classifier on the **job's TF-IDF only** shows how much the job side carries.

In [ ]:
def grouped_oof(df, text_col, group_col, C=1.0):
    y = df.good.to_numpy(); oof = np.zeros(len(df))
    for tr, te in GroupKFold(5).split(df, y, df[group_col]):
        vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=50_000)
        Xtr = vec.fit_transform(df.iloc[tr][text_col]); Xte = vec.transform(df.iloc[te][text_col])
        clf = LogisticRegression(C=C, max_iter=2000, class_weight='balanced').fit(Xtr, y[tr])
        oof[te] = clf.predict_proba(Xte)[:, 1]
    return oof

def per_job_auc(df, scores):
    d = df.assign(s=scores); out = []
    for _, g in d.groupby('job_id'):
        if 0 < g.good.sum() < len(g):
            out.append(roc_auc_score(g.good, g.s))
    return np.array(out)

def boot(v, n=2000):
    r = np.random.default_rng(SEED)
    return np.percentile([r.choice(v, len(v)).mean() for _ in range(n)], [2.5, 97.5])

fit['resume_text_only'] = grouped_oof(fit, 'resume_text', 'resume_id')
fit['job_text_only'] = grouped_oof(fit, 'job_text', 'job_id')

tot = all_df.groupby('resume_id').relevance.agg(['sum', 'count'])
gm = all_df.relevance.mean()
fit['resume_prior'] = [((tot.loc[r, 'sum'] - v) / (tot.loc[r, 'count'] - 1)) if tot.loc[r, 'count'] > 1 else gm
                       for r, v in zip(fit.resume_id, fit.relevance)]

auc_rows = []
for col, label in [('resume_prior', 'Resume-prior (identity, leave-one-job-out)'),
                   ('resume_text_only', 'Resume TEXT only, unseen resumes'),
                   ('job_text_only', 'Job TEXT only, unseen jobs')]:
    a = per_job_auc(fit, fit[col]); lo, hi = boot(a)
    pooled = roc_auc_score(fit.good, fit[col])
    auc_rows.append({'scorer': label, 'jobs': len(a), 'per-job AUC good-vs-potential': round(a.mean(), 3),
                     'CI': f'[{lo:.3f}, {hi:.3f}]', 'pooled AUC': round(pooled, 3)})
auc_tab = pd.DataFrame(auc_rows)
auc_tab

**How to read it:** per-job AUC ranks candidates *within one job*, so the job-text scorer is constant inside a job and should sit near 0.5 there. Its *pooled* AUC shows whether some jobs simply hand out more Goods. If the resume-text scorer is well above 0.5 on unseen resumes, then **the resume's own wording predicts the label, whatever the job is.**

### What words does the label reward?
Fit on all fit pairs (for interpretation only, not evaluation). Each resume is counted once per fit pair, so resumes with many pairs weigh more.

In [ ]:
vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=5, sublinear_tf=True, max_features=50_000)
X = vec.fit_transform(fit.resume_text)
clf = LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced').fit(X, fit.good)
terms = np.array(vec.get_feature_names_out()); w = clf.coef_[0]
top = pd.DataFrame({'toward GOOD': terms[np.argsort(-w)[:30]], 'toward POTENTIAL': terms[np.argsort(w)[:30]]})
top

## 4. Simple resume properties vs the label

Every feature is computed from the resume alone. For each one we report
- **Spearman ρ** with the resume's share of Good (resumes with at least 2 fit pairs), and
- **per-job AUC** for Good vs Potential when that single feature is used as the score.

In [ ]:
def years_exp(t):
    ys = [int(y) for y in re.findall(r'(\d{1,2})\+?\s*(?:years|yrs)', t.lower()) if int(y) <= 45]
    return max(ys) if ys else 0

def degree_level(t):
    t = t.lower()
    if re.search(r'\bph\.?d\b|doctorate', t): return 3
    if re.search(r"\bmaster|\bm\.?s\.?c?\b|\bmba\b|\bm\.?tech\b", t): return 2
    if re.search(r"\bbachelor|\bb\.?s\.?c?\b|\bb\.?tech\b|\bb\.?e\.?\b|\bb\.?a\.?\b", t): return 1
    return 0

feat_fns = {'word count': lambda t: len(t.split()),
            'max years of experience stated': years_exp,
            'degree level (0-3)': degree_level,
            'distinct words': lambda t: len(set(t.lower().split())),
            'digits (numbers, dates)': lambda t: sum(c.isdigit() for c in t),
            'lines / bullets': lambda t: t.count('\n') + t.count('•')}
res_text = all_df.drop_duplicates('resume_id').set_index('resume_id').resume_text
feats = pd.DataFrame({k: res_text.map(f) for k, f in feat_fns.items()})

share_good = fit.groupby('resume_id').good.agg(['mean', 'size'])
share_good = share_good[share_good['size'] >= 2]['mean']
rows = []
for k in feat_fns:
    rho, p = spearmanr(feats.loc[share_good.index, k], share_good)
    a = per_job_auc(fit, fit.resume_id.map(feats[k]).to_numpy())
    rows.append({'feature': k, 'median': feats[k].median(), 'Spearman rho vs share Good': round(rho, 3),
                 'p': float(f'{p:.2g}'), 'per-job AUC good-vs-potential': round(np.nanmean(a), 3)})
feat_tab = pd.DataFrame(rows)
feat_tab

## 5. Summary for the paper and save

In [ ]:
g = decomp[decomp.target == 'good'].set_index('factors')
w = decomp[decomp.target == 'good (within job)'].iloc[0]
print('RQ3 summary (Good vs Potential among fit pairs):')
print(f"  between jobs: job identity explains {g.loc['job only', 'in-sample R2']:.2f}, resume identity "
      f"{g.loc['resume only', 'in-sample R2']:.2f}")
print(f"  within jobs: resume identity explains {w['in-sample R2']:.2f} (in-sample), {w['CV R2 (new pairs)']:.2f} for new pairs")
print(f'  label purity {obs:.3f} vs {null.mean():.3f} expected if labels followed the job only (p = {p_purity:.4f}); '
      f'{share_pure:.0%} of resumes are all-Good or all-Potential')
for _, r in auc_tab.iterrows():
    print(f"  {r['scorer']}: per-job AUC {r['per-job AUC good-vs-potential']} {r['CI']}")

decomp.to_csv(f'{RESULTS}/06b_variance_decomposition.csv', index=False)
auc_tab.to_csv(f'{RESULTS}/06b_text_only_auc.csv', index=False)
top.to_csv(f'{RESULTS}/06b_top_terms.csv', index=False)
feat_tab.to_csv(f'{RESULTS}/06b_surface_features.csv', index=False)
pd.DataFrame({'observed': [obs], 'null_mean': [null.mean()], 'p': [p_purity], 'share_pure': [share_pure],
              'resumes': [n_res]}).to_csv(f'{RESULTS}/06b_purity.csv', index=False)
print('saved')